# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedasker1/FlyRank_Repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

*Finding 1: "Content with a higher word count consistently sees less traffic decay."
Methodology Question: Is this an observed correlation, or is it implying causality? Furthermore, does the validation design control for the 'content type' or 'intent', since informational pages naturally have more words and might decay differently than product pages?

Finding 2: "The model identifies decaying pages with 85% accuracy."
Methodology Question: What is the base rate of the 'decay' label? If 80% of pages aren't decaying, a naive model guessing "no decay" gets 80% accuracy. Also, was the validation time-aware (trained on past data, tested on future data), or a random split that might leak future temporal patterns?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Review focus: Checking for causal language vs. correlational evidence, and ensuring validation splits reflect real-world time and group constraints.")

Review focus: Checking for causal language vs. correlational evidence, and ensuring validation splits reflect real-world time and group constraints.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

*A standard random split (train_test_split) is dishonest for this dataset because multiple pages belong to the same client. If we split randomly, the model might memorize client-specific traffic baselines rather than learning universal decay signals. The "honest" split uses GroupShuffleSplit on client_id to ensure the model is tested on entirely unseen clients.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# 1. Environment Setup
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# 2. Load Data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

features = ['content_age_days', 'impressions_90d', 'ctr', 'avg_position', 'word_count']
df_clean = df.dropna(subset=features + ['client_id', 'is_declining']).copy()

X = df_clean[features]
y = df_clean['is_declining']
groups = df_clean['client_id']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk_labels = np.asarray(labels)[order[:k]]
    return topk_labels.mean()

# --- BEFORE: Dishonest Random Split ---
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(X, y, test_size=0.25, random_state=42)
rf_rand = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42).fit(X_tr_rand, y_tr_rand)
scores_rand = rf_rand.predict_proba(X_te_rand)[:, 1]
p50_rand = precision_at_k(scores_rand, y_te_rand, k=50)

# --- AFTER: Honest Grouped Split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_grp, X_te_grp = X.iloc[train_idx], X.iloc[test_idx]
y_tr_grp, y_te_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42).fit(X_tr_grp, y_tr_grp)
scores_grp = rf_grp.predict_proba(X_te_grp)[:, 1]
p50_grp = precision_at_k(scores_grp, y_te_grp, k=50)

print("--- Split Comparison (Precision@50) ---")
print(f"Dishonest (Random Split) Precision@50: {p50_rand:.3f} (Inflated due to data leakage across clients)")
print(f"Honest (Grouped Split) Precision@50:   {p50_grp:.3f} (The true, earned metric on unseen clients)")

--- Split Comparison (Precision@50) ---
Dishonest (Random Split) Precision@50: 0.860 (Inflated due to data leakage across clients)
Honest (Grouped Split) Precision@50:   0.580 (The true, earned metric on unseen clients)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

*Checking the final feature set to ensure no target-derived variables (like trend_pct or trend_direction) or product flags slipped into the training data. If a feature perfectly predicts the target, it's usually a leak, not magic.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check if any strictly prohibited columns made it into our feature set X
prohibited_leaks = ['trend_pct', 'trend_direction', 'is_declining', 'future_impressions', 'health_score']
leaks_found = [col for col in X.columns if col in prohibited_leaks]

print("--- Leakage Audit ---")
if not leaks_found:
    print("PASS: No direct label-derived columns or product flags found in the feature set.")
else:
    print(f"FAIL: The following leaky columns were detected: {leaks_found}")

# Sanity check: verify no single feature has an unrealistic correlation with the target (> 0.8)
correlations = df_clean[features].corrwith(df_clean['is_declining']).abs()
suspicious = correlations[correlations > 0.8]

if suspicious.empty:
    print("PASS: No suspiciously high correlations detected. Features appear to be observable signals.")
else:
    print(f"WARNING: Suspiciously high correlations found, investigate for leakage:\n{suspicious}")

--- Leakage Audit ---
PASS: No direct label-derived columns or product flags found in the feature set.
PASS: No suspiciously high correlations detected. Features appear to be observable signals.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

*Bold/Unsafe Claim: "The model successfully predicts exactly which pages will lose traffic and proves that updating older content will rescue a site's search visibility."

Safe/Honest Rewrite: "We observed a directional link between certain historical metrics (like staleness and low CTR) and performance decay. This model acts as a decision-support tool, ranking pages by their measured risk of decline to help prioritize the content team's review queue."*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rule of thumb applied: Swapped 'predicts exactly' for 'measured risk', and 'proves' for 'observed a directional link'.")

Rule of thumb applied: Swapped 'predicts exactly' for 'measured risk', and 'proves' for 'observed a directional link'.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.